In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MyApp")
    .master("spark://10.1.11.5:7077")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "4")
    .config("spark.executor.instances", "2")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2")  # 4.0.2 + Scala 2.13
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Packages:", spark.sparkContext.getConf().get("spark.jars.packages", "NOT SET"))

In [ ]:
data = [
    ("Abraham", "Apples",  5),
    ("Jacob",   "Bananas", 3),
    ("Jacob",   "Oranges", 2),
    ("Benjamin","Apples",  2),
    ("David",   "Bananas", 2),
    ("David",   "Oranges", 5),
    ("Esau",    "Apples",  3),
    ("Esau",    "Bananas", 5),
    ("Ishmael", "Oranges", 4),
    ("Samael",  "Apples",  4),
    ("Samael",  "Bananas", 1),
    ("Samael",  "Oranges", 1),
]
columns = ["Customer", "Item", "Quantity"]
df = spark.createDataFrame(data, columns)

In [ ]:
df.show()         # Print rows (like pandas df.head())
print(df.count()) # Row count → 12
df.printSchema()  # Column types (like pandas df.info())

In [ ]:
from pyspark.sql.functions import col, sum

# APPROACH 1: filter first → then agg
total_apples = (df
    .filter(col("Item") == "Apples")
    .agg(sum("Quantity").alias("Total_Apples"))
    .collect()[0]["Total_Apples"])
print(f"Total Apples sold: {total_apples}")

# APPROACH 2: groupBy → agg → filter
total_apples_2 = (df
    .groupBy("Item")
    .agg(sum("Quantity").alias("Total_Quantity"))
    .filter(col("Item") == "Apples")
    .collect()[0]["Total_Quantity"])
print(f"Total Apples sold (approach 2): {total_apples_2}")

# APPROACH 3: SQL — filter in WHERE
df.createOrReplaceTempView("purchases")
total_apples_3 = spark.sql("""
    SELECT SUM(Quantity) AS Total_Apples
    FROM purchases
    WHERE Item = 'Apples'
""").collect()[0]["Total_Apples"]
print(f"Total Apples sold (approach 3): {total_apples_3}")

# APPROACH 4: SQL — filter in HAVING
total_apples_4 = spark.sql("""
    SELECT Item, SUM(Quantity) AS Total_Quantity
    FROM purchases
    GROUP BY Item
    HAVING Item = 'Apples'
""").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 4): {total_apples_4}")

In [ ]:
price_data = [("Apples", 1.0), ("Bananas", 0.5), ("Oranges", 0.8)]
price_columns = ["Item", "Price"]
price_df = spark.createDataFrame(price_data, price_columns)
price_df.show()

In [ ]:
from pyspark.sql.functions import expr

# Join on "Item" column (inner join)
joined_df = df.join(price_df, on="Item", how="inner")

# Add computed column: Total_Cost = Quantity * Price
joined_df = joined_df.withColumn("Total_Cost", expr("Quantity * Price"))

# Filter Jacob and sum his costs
jacob_total = (joined_df
    .filter(col("Customer") == "Jacob")
    .agg(sum("Total_Cost").alias("Jacob_Total"))
    .collect()[0]["Jacob_Total"])
print(f"Jacob's total grocery spending: ${jacob_total:.2f}")

In [ ]:
hc = spark.sparkContext._jsc.hadoopConfiguration()

# S3A library for AWS/MinIO
hc.set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")

# MinIO endpoint and credentials
hc.set("fs.s3a.endpoint",   "http://10.1.11.5:9000")
hc.set("fs.s3a.access.key", "test")
hc.set("fs.s3a.secret.key", "hcmuthpcc")

# Path-style access (required for MinIO)
hc.set("fs.s3a.path.style.access", "true")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hc.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# Magic committer — better than default FileOutputCommitter
hc.set("fs.s3a.committer.name", "magic")
hc.set("fs.s3a.committer.magic.partitioned.enabled", "true")
hc.set("fs.s3a.fast.upload",        "true")
hc.set("fs.s3a.fast.upload.buffer", "disk")

# Optional: overwrite and abort pending
hc.set("fs.s3a.committer.staging.conflict-mode",          "replace")
hc.set("fs.s3a.committer.staging.abort.pending.uploads",  "true")

In [ ]:
df.write.mode("overwrite").csv(
    "s3a://big-data/test/jacob.csv",
    header=True
)


In [ ]:
import socket
for host, port in [("10.1.1.10", 30090), ("10.1.1.27", 30091), ("10.1.1.203", 30092)]:
    try:
        socket.create_connection((host, port), timeout=3)
        print(f"{host}:{port} reachable")
    except Exception as e:
        print(f"{host}:{port} FAILED — {e}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType
)

BROKERS = "10.1.1.10:30090,10.1.1.27:30091,10.1.1.203:30092"
# ── Define schemas ──────────────────────────────────────────────────
ratings_schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("rating",    DoubleType()),
    StructField("timestamp", LongType()),
])

movies_schema = StructType([
    StructField("movieId", IntegerType()),
    StructField("title",   StringType()),
    StructField("genres",  StringType()),
])

tags_schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("tag",       StringType()),
    StructField("timestamp", LongType()),
])

# ── Helper: read topic, cast binary → string → parse JSON ───────────
def read_topic(topic, schema):
    raw = (spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", BROKERS)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load())
    # FIX: value is binary → cast to string → parse as JSON
    return (raw
        .selectExpr("CAST(value AS STRING) AS json_str")
        .select(from_json(col("json_str"), schema).alias("data"))
        .select("data.*"))

df_ratings = read_topic("ratings", ratings_schema)
df_movies  = read_topic("movies",  movies_schema)
df_tags    = read_topic("tags",    tags_schema)

df_ratings.show(3)
df_movies.show(3)
df_tags.show(3)

In [ ]:
from pyspark.sql.functions import avg, count

# Group by movieId → avg rating + count
movie_stats = (df_ratings
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("rating_count")
    )
    .filter(col("rating_count") > 30)          # only movies with >30 ratings
    .orderBy(col("avg_rating").desc())            # sort highest first
    .limit(5))

# Join with movie titles
top5_movies = movie_stats.join(df_movies, on="movieId", how="left")
top5_movies.select("movieId", "title", "avg_rating", "rating_count").show(truncate=False)

In [ ]:
# Join tags with ratings on movieId
tags_with_ratings = df_tags.join(df_ratings, on="movieId", how="inner")

# Group by tag → compute average rating of tagged movies
worst_tags = (tags_with_ratings
    .groupBy("tag")
    .agg(avg("rating").alias("avg_rating"))
    .orderBy(col("avg_rating").asc())   # lowest first
    .limit(5))

worst_tags.show(truncate=False)

In [ ]:
# Step 1: collect the 5 worst tag strings
worst_tag_list = [row["tag"] for row in worst_tags.collect()]
print("Worst tags:", worst_tag_list)

# Step 2: filter tags table to only those worst tags
filtered_tags = df_tags.filter(col("tag").isin(worst_tag_list))

# Step 3: join with ratings to get each movie's ratings
tagged_movies_ratings = filtered_tags.join(df_ratings, on="movieId", how="inner")

# Step 4: per-movie average rating AND how many movies share each tag
movie_tag_stats = (tagged_movies_ratings
    .groupBy("tag", "movieId")
    .agg(
        avg("rating").alias("movie_avg_rating"),
        count("rating").alias("n_ratings")
    )
    .orderBy("tag", col("movie_avg_rating").asc()))

# Step 5: join with movie titles for readability
movie_tag_stats = movie_tag_stats.join(
    df_movies.select("movieId", "title"), on="movieId", how="left"
)
movie_tag_stats.show(30, truncate=False)

# Step 6: how many distinct movies have each worst tag?
tag_movie_count = (filtered_tags
    .groupBy("tag")
    .agg(count("movieId").alias("movie_count")))
print("Movies per worst tag:")
tag_movie_count.show(truncate=False)